# Microsoft Foundry - AI Agent Telemetry Notebook Guide

This notebook demonstrates how to create and interact with an AI agent using the **Azure AI Projects SDK**. The agent is configured as a storytelling agent that crafts engaging stories based on user prompts. It also enables **OpenTelemetry tracing** to send agent telemetry to Application Insights and the `AppDependencies` table in Log Analytics.

---

## Prerequisites

Before running this notebook, ensure the following are in place:

### 1. Azure CLI Installed
The notebook uses `DefaultAzureCredential`, which relies on Azure CLI for local authentication.

- **Install via winget (Windows):**
  ```powershell
  winget install --id Microsoft.AzureCLI -e --accept-source-agreements --accept-package-agreements
  ```
- **Other platforms / manual install:** [https://aka.ms/installazurecli](https://aka.ms/installazurecli)

### 2. Logged in via Azure CLI with Appropriate Permissions
After installing, sign in and verify your session:
```bash
az login
az account show
```
Your Entra ID identity must have **Contributor** (or equivalent) access to the Microsoft Foundry project and the associated Application Insights resource.

### 3. Microsoft Foundry Project with Application Insights
You need a project provisioned in **Microsoft Foundry** that is connected to an **Application Insights** instance backed by a **Log Analytics workspace**. The notebook retrieves the Application Insights connection string from the project at runtime to enable telemetry export.

---

## 0. Create or Reuse Virtual Environment & Register Kernel
- Create (or reuse) a Python virtual environment to isolate dependencies for this project, then install `ipykernel` and register it as a notebook kernel.
- Once this cell is completed, select **Kernel** and choose your new `.venv` environment.

In [ ]:
import os
import subprocess
import sys

venv_dir = os.path.join(os.getcwd(), ".venv")

# Create the virtual environment (safe to run repeatedly)
subprocess.check_call([sys.executable, "-m", "venv", venv_dir])

# Resolve the venv Python path per OS, then use "python -m pip" for reliability
venv_python = (
    os.path.join(venv_dir, "Scripts", "python.exe")
    if os.name == "nt"
    else os.path.join(venv_dir, "bin", "python")
)

# Install / upgrade ipykernel inside the venv
subprocess.check_call([venv_python, "-m", "pip", "install", "--upgrade", "ipykernel"])

# Register the venv as a Jupyter kernel
subprocess.check_call([
    venv_python,
    "-m",
    "ipykernel",
    "install",
    "--user",
    "--name",
    "ai-agent-demo",
    "--display-name",
    "AI Agent Demo (.venv)",
])

print(f"Virtual environment created or reused at {venv_dir}")
print("Select the 'AI Agent Demo (.venv)' kernel in the top-right kernel picker, then continue.")

## 1. Install Dependencies

Install the required Python packages:
- **azure-ai-projects** — SDK for working with Microsoft Foundry projects and agents
- **azure-identity** — Provides `DefaultAzureCredential` for seamless Azure authentication
- **azure-monitor-opentelemetry** — All-in-one Azure Monitor OpenTelemetry distro (configures exporters, tracer providers, etc.)
- **opentelemetry-sdk** — Core OpenTelemetry SDK for instrumentation and tracing
- **azure-core-tracing-opentelemetry** — Azure SDK tracing plugin for OpenTelemetry

In [ ]:
import json
import subprocess
import sys

outdated = subprocess.check_output(
    [sys.executable, "-m", "pip", "list", "--outdated", "--format=json"],
    text=True,
)
if any(pkg["name"].lower() == "pip" for pkg in json.loads(outdated)):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--upgrade", "pip"])

# Keep Azure Monitor exporter current to avoid OpenTelemetry import mismatches.
%pip --disable-pip-version-check install --upgrade --pre azure-monitor-opentelemetry-exporter
%pip --disable-pip-version-check install --pre "azure-ai-projects>=2.0.0b4"
%pip --disable-pip-version-check install azure-identity azure-monitor-opentelemetry azure-core-tracing-opentelemetry

## 2. Import Libraries

Import the necessary classes from the Azure SDKs:
- `DefaultAzureCredential` — Automatically picks up credentials from your environment (Azure CLI, managed identity, etc.)
- `AIProjectClient` — Client for interacting with your Microsoft Foundry project
- `PromptAgentDefinition` — Defines the agent's model and behavior instructions
- `AIProjectInstrumentor` — Instruments all `azure-ai-projects` and OpenAI SDK operations for tracing (imported in Section 3.5)
- `configure_azure_monitor` — One-liner to set up Azure Monitor as the OpenTelemetry export backend

In [ ]:
import sys, platform

from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition
from azure.ai.projects.telemetry import AIProjectInstrumentor

print("✅ All libraries imported successfully")
print(f"   💻 Platform: {platform.system()} {platform.release()}")
print(f"   🐍 Python:   {sys.version.split()[0]}")

## 3. Configure the Project Client

- Set the Microsoft Foundry project endpoint.
- Create an authenticated `AIProjectClient` instance.

Update the `user_endpoint` to match your own project URL.

In [ ]:

import logging
import os
import shutil
import subprocess

user_endpoint = "https://zolab1702-ai-foundry.services.ai.azure.com/api/projects/zolab1702-ai-fndry-proj"

# ---------------------------------------------------------------------------
# Helper: install Azure CLI via winget (Windows only)
# ---------------------------------------------------------------------------
def _install_az_cli() -> bool:
    """Attempt to install Azure CLI using winget. Returns True on success."""
    if os.name != "nt":
        print("❌ Automatic install via winget is only supported on Windows.")
        print("   👉 Install Azure CLI from: https://aka.ms/installazurecli")
        return False

    if not shutil.which("winget"):
        print("❌ winget is not available on this system.")
        print("   👉 Install Azure CLI manually from: https://aka.ms/installazurecli")
        return False

    print("📦 Installing Azure CLI via winget … (this may take a few minutes)")
    result = subprocess.run(
        ["winget", "install", "--id", "Microsoft.AzureCLI", "-e", "--accept-source-agreements", "--accept-package-agreements"],
        capture_output=True, text=True, timeout=300,
    )
    if result.returncode == 0:
        print("✅ Azure CLI installed successfully.")
        # Refresh PATH so the current process can find az.cmd immediately
        az_default = os.path.join(os.environ.get("ProgramFiles", r"C:\Program Files"), "Microsoft SDKs", "Azure", "CLI2", "wbin")
        if os.path.isdir(az_default) and az_default not in os.environ.get("PATH", ""):
            os.environ["PATH"] = az_default + os.pathsep + os.environ["PATH"]
            print(f"   Added {az_default} to PATH for this session.")
        return True
    else:
        print(f"❌ winget install failed (exit code {result.returncode}).")
        if result.stderr:
            print(f"   {result.stderr.strip()}")
        print("   👉 Try installing manually from: https://aka.ms/installazurecli")
        return False

# ---------------------------------------------------------------------------
# Helper: check for Azure CLI and attempt interactive login if needed
# ---------------------------------------------------------------------------
def _ensure_az_login() -> bool:
    """Return True if Azure CLI is available and the user is logged in."""
    az_exe = "az.cmd" if os.name == "nt" else "az"

    if not shutil.which(az_exe):
        print("⚠️  Azure CLI is NOT installed.")
        if not _install_az_cli():
            return False
        # Re-check after install
        if not shutil.which(az_exe):
            print("❌ Azure CLI still not found on PATH after install.")
            print("   Please restart VS Code so the updated PATH takes effect, then re-run this cell.")
            return False

    print("🔍 Azure CLI found on PATH — checking login status …")

    # Quick check: is there already a valid session?
    check = subprocess.run(
        [az_exe, "account", "show"],
        capture_output=True, text=True, timeout=15,
    )
    if check.returncode == 0:
        print("✅ Already logged in to Azure CLI.")
        return True

    # Not logged in → run interactive login
    print("⚠️  No active Azure CLI session detected. Launching 'az login' …")
    print("   A browser window should open — please sign in with your Azure account.\n")
    login = subprocess.run([az_exe, "login"], capture_output=True, text=True, timeout=120)
    if login.returncode == 0:
        if login.stdout:
            print(login.stdout.strip())
        print("\n✅ Azure CLI login succeeded.")
        return True
    else:
        print(f"\n❌ 'az login' did not complete successfully (exit code {login.returncode}).")
        if login.stderr:
            print(f"   {login.stderr.strip()}")
        print("   Please run 'az login' manually in a terminal, then re-run this cell.")
        return False

# ---------------------------------------------------------------------------
# Authenticate
# ---------------------------------------------------------------------------
credential = DefaultAzureCredential()

# Temporarily enable azure.identity INFO logging to surface which credential is used
_identity_handler = logging.StreamHandler()
identity_logger = logging.getLogger("azure.identity")
identity_logger.addHandler(_identity_handler)
identity_logger.setLevel(logging.INFO)

try:
    # Probe the credential to fail fast if nothing works
    _ = credential.get_token("https://management.azure.com/.default")

except Exception as auth_ex:
    print(f"⚠️  DefaultAzureCredential failed: {type(auth_ex).__name__}")
    print("   Falling back to Azure CLI authentication …\n")

    if not _ensure_az_login():
        raise SystemExit(
            "🛑 Cannot proceed without valid Azure credentials. "
            "Please install/login to Azure CLI and re-run this cell."
        )

    # Retry after successful az login
    credential = DefaultAzureCredential()
    _ = credential.get_token("https://management.azure.com/.default")

finally:
    # Clean up the INFO handler so background token-refresh messages don't
    # keep writing to stderr and make the notebook cell spin forever.
    identity_logger.removeHandler(_identity_handler)
    identity_logger.setLevel(logging.WARNING)

project_client = AIProjectClient(
    endpoint=user_endpoint,
    credential=credential,
)

print("\n✅ AIProjectClient configured")
print(f"   🌐 Endpoint: {user_endpoint}")
print("   🔐 Auth:     DefaultAzureCredential")

# ---------------------------------------------------------------------------
# Optional: surface which credential + identity is in use
# ---------------------------------------------------------------------------
def _run_command(command: list[str]) -> str | None:
    try:
        result = subprocess.run(
            command,
            capture_output=True,
            text=True,
            timeout=10,
            check=False,
        )
        if result.returncode == 0:
            value = (result.stdout or "").strip()
            return value or None
    except Exception:
        pass
    return None

def _resolve_identity_hint(credential_name: str) -> str | None:
    if credential_name == "AzureCliCredential":
        az_exe = "az.cmd" if os.name == "nt" else "az"
        return _run_command([az_exe, "account", "show", "--query", "user.name", "-o", "tsv"])
    if credential_name == "AzurePowerShellCredential":
        powershell_cmd = "$ctx = Get-AzContext; if ($ctx -and $ctx.Account) { $ctx.Account.Id }"
        return _run_command(["pwsh", "-NoProfile", "-Command", powershell_cmd])
    return None

try:
    selected_credential = getattr(credential, "_successful_credential", None)
    if selected_credential is not None:
        credential_name = selected_credential.__class__.__name__
        print(f"   🔐 Credential used: {credential_name}")

        identity_hint = _resolve_identity_hint(credential_name)
        if identity_hint:
            print(f"   👤 Signed-in account: {identity_hint}")
        else:
            print("   👤 Signed-in account: not available for this credential type")
    else:
        print("   Credential Type: check azure.identity INFO logs above")
except Exception as ex:
    print(f"   Credential probe skipped: {type(ex).__name__}: {ex}")


## 3.5 Enable Telemetry

Configure tracing per the [Azure AI Projects SDK tracing guide](https://github.com/Azure/azure-sdk-for-python/tree/main/sdk/ai/azure-ai-projects#tracing):
1. **`configure_azure_monitor`** — Sets up the full OpenTelemetry pipeline (TracerProvider, exporter, span processors) to send traces to Application Insights
2. **`AIProjectInstrumentor`** — Instruments all `azure-ai-projects` SDK operations (agent create/version, list, etc.) and automatically instruments OpenAI responses/conversations operations

### MSFT Learn MCP (before agent creation)
- MSFT Learn MCP URL: `https://learn.microsoft.com/api/mcp`
- The next code cell prints this URL, creates `mcp_tool_spec`, and makes it available for agent creation.

**Note:**
- `AZURE_EXPERIMENTAL_ENABLE_GENAI_TRACING=true` must be set **before** calling `instrument()`.
- Content recording is controlled by `OTEL_INSTRUMENTATION_GENAI_CAPTURE_MESSAGE_CONTENT`.

In [ ]:
from opentelemetry import trace

# Must be set BEFORE importing/calling monitor + instrument()
os.environ["AZURE_EXPERIMENTAL_ENABLE_GENAI_TRACING"] = "true"
os.environ["OTEL_INSTRUMENTATION_GENAI_CAPTURE_MESSAGE_CONTENT"] = "true"

# Enable end-to-end correlation between client and service spans
os.environ["AZURE_TRACING_GEN_AI_ENABLE_TRACE_CONTEXT_PROPAGATION"] = "true"
os.environ["AZURE_TRACING_GEN_AI_TRACE_CONTEXT_PROPAGATION_INCLUDE_BAGGAGE"] = "true"

# Get the Application Insights connection string from the Foundry project
application_insights_connection_string = project_client.telemetry.get_application_insights_connection_string()

# Note: The connection string is required to configure the OpenTelemetry exporter, but the underlying Azure Monitor 
# exporter will also look for it in the AZURE_MONITOR_CONNECTION_STRING environment variable as a fallback, 
# so it's not strictly necessary to pass it here if it's already set in the environment.
from azure.monitor.opentelemetry import configure_azure_monitor

# Configure Azure Monitor as the tracing backend (one-liner per SDK docs)
configure_azure_monitor(connection_string=application_insights_connection_string)

# Instrument azure-ai-projects SDK + OpenAI responses/conversations automatically
AIProjectInstrumentor().instrument(enable_content_recording=True)

tracer = trace.get_tracer(__name__)

print(f"Tracing enabled → Application Insights (connection string retrieved from project)")

### 3.6 Configure MSFT Learn MCP Tool

Run the next cell to create `mcp_tool_spec` for use in Step 4.

- MSFT Learn MCP documentation: [Azure MCP Server docs](https://learn.microsoft.com/azure/developer/azure-mcp-server/)
- MSFT Learn MCP URL: `https://learn.microsoft.com/api/mcp`
- It prints the Microsoft Learn MCP endpoint for quick verification.
- It builds an `MCPTool` spec with server label `msft-learn`.
- Step 4 uses this object via `tools=[mcp_tool_spec]` when creating the agent.

In [ ]:
# ---------------------------------------------------------------------------
# MSFT Learn MCP Tool Spec (used by Step 4 agent creation)
# ---------------------------------------------------------------------------
from azure.ai.projects.models import MCPTool

msft_learn_mcp_url = "https://learn.microsoft.com/api/mcp"
print(f"MSFT Learn MCP URL: {msft_learn_mcp_url}")

mcp_tool_spec = MCPTool(
    server_label="msft-learn",
    server_url=msft_learn_mcp_url,
)



## 4. Create the Agent

Define and create a versioned agent. Replace `<your-agent-name>` and `<your-model-deployment-name>` with your actual values.

- `create_version` will create a new agent or bump the version if the parameters have changed.
- The agent is given a **storytelling** persona via the `instructions` field.
- The MSFT Learn MCP tool is available to the agent via `mcp_tool_spec` in `tools=[mcp_tool_spec]`.

In [ ]:
agent_name = "ZoDEfendersAgent-1702"
model_name = "gpt-4.1-mini"
agent_instructions = (
    "You are an illuminating cybernetic storytelling agent. "
    "You craft engaging 6-sentence stories based on user prompts and context. "
    "Be sure to use vivid language and creative scenarios to captivate the reader. "
    "Lastly, use the full names of any characters you create in the story."
)

# Wrap agent creation in a tracing span for end-to-end observability
with tracer.start_as_current_span("agent-creation") as span:
    span.set_attribute("agent.name", agent_name)
    span.set_attribute("gen_ai.request.model", model_name)

    # Creates an agent, bumps the agent version if parameters have changed
    agent = project_client.agents.create_version(
        agent_name=agent_name,
        definition=PromptAgentDefinition(
            model=model_name,
            instructions=agent_instructions,
            tools=[mcp_tool_spec],
        ),
    )
    span.set_attribute("agent.version", agent.version)
    span.set_attribute("agent.id", agent.id)

print(f"Agent created: {agent.name} (id: {agent.id}, version: {agent.version})")

## 5. Query the Agent

Use the project’s OpenAI-compatible client to send a prompt to the agent and retrieve a response.

- The agent is referenced by name using `agent_reference`.
- Each response is appended to `stories.json` as a new object in the array.
- You can access individual stories via `stories[0]`, `stories[1]`, etc.

In [ ]:
import json
from datetime import datetime, timezone
from pathlib import Path

openai_client = project_client.get_openai_client()

user_prompt = ( 
    "Write about a Cloud Solutions Architect - Security by the name of 'Azure Zo' in Frankfurt, Germany. "
    "He is a man who moonlights as a cybernetic superhero called 'Die DEfender', protector of the digital realm. "
    "Include a plot twist in the last sentence. Lastly, use what tools you have available to tell me about Microsoft Foundry."
)

# Wrap the agent query in a tracing span for end-to-end observability
with tracer.start_as_current_span("agent-query") as span:
    span.set_attribute("agent.name", agent.name)
    span.set_attribute("gen_ai.request.model", model_name)
    span.set_attribute("gen_ai.prompt", user_prompt)

    # Reference the agent to get a response
    response = openai_client.responses.create(
        input=[{"role": "user", "content": user_prompt}],
        extra_body={"agent_reference": {"name": agent.name, "id": agent.id, "type": "agent_reference"}},
    )

    span.set_attribute("gen_ai.completion", response.output_text[:500])
    span.set_attribute("gen_ai.response.id", response.id)

print(f"Response output: {response.output_text}")

def append_story(stories_path: Path, payload: dict) -> int:
    try:
        with stories_path.open("r", encoding="utf-8") as file:
            stories = json.load(file)
        if not isinstance(stories, list):
            stories = []
    except (FileNotFoundError, json.JSONDecodeError):
        stories = []

    next_id = max((item.get("id", 0) for item in stories), default=0) + 1
    payload["id"] = next_id
    stories.append(payload)

    with stories_path.open("w", encoding="utf-8") as file:
        json.dump(stories, file, indent=2, ensure_ascii=False)

    return next_id

stories_file = Path("stories.json")
story_id = append_story(
    stories_file,
    {
        "timestamp": datetime.now(timezone.utc).isoformat(),
        "agent": agent.name,
        "model": model_name,
        "prompt": user_prompt,
        "story": response.output_text,
    },
)

print(f"\nStory #{story_id} saved to {stories_file}")

## 6. Validate Traces in Log Analytics

Use this section to validate that telemetry from the notebook is landing in the Log Analytics workspace hosted in the Security subscription.

### Notes
- Workspace resource ID: `/subscriptions/<subscription-id>/resourceGroups/Sentinel/providers/Microsoft.OperationalInsights/workspaces/<Log-Analytics-Workspace-Name>`
- The Azure CLI extension command `az monitor log-analytics query` may fail in some environments due to extension/runtime mismatch.
- The code cell below uses `az rest` against `api.loganalytics.io`, which is a reliable fallback.

### Query outputs
- The first query gives an end-to-end joined view of dependencies and traces by `OperationId`.
- The second query gives a compact runs-only trend with call volume, failures, and latency percentiles.

In [ ]:
import json
import urllib.error
import urllib.request

from azure.identity import DefaultAzureCredential

workspace_customer_id = "7e9298ab-22e6-4a82-a53e-c5ed7faee977"

# End-to-end view including requested fields: agent name, model, system message, host, location
kql_end_to_end = r"""
let lookback = 6h;
AppDependencies
| where TimeGenerated > ago(lookback)
| where Name in ("agent-query", "responses") or Name startswith "chat "
| extend
    agent_name = coalesce(
        tostring(Properties["agent.name"]),
        tostring(split(tostring(Properties["gen_ai.agent.id"]), ":")[0]),
        ""
    ),
    model_used = coalesce(
        tostring(Properties["gen_ai.request.model"]),
        tostring(Properties["gen_ai.response.model"]),
        ""
    ),
    system_message = coalesce(
        extract(@'"role"\s*:\s*"system".*?"content"\s*:\s*"([^\"]+)"', 1, tostring(Properties["gen_ai.input.messages"])),
        ""
    ),
    computer = tostring(AppRoleInstance),
    location = coalesce(
        tostring(Properties["_workspace_location"]),
        tostring(Properties["cloud.region"]),
        ""
    )
| project
    TimeGenerated,
    Name,
    agent_name,
    model_used,
    system_message,
    computer,
    location,
    Success,
    DurationMs,
    OperationId,
    Prompt = tostring(Properties["gen_ai.prompt"]),
    Completion = tostring(Properties["gen_ai.completion"])
| order by TimeGenerated desc
| take 50
""".strip()

# Runs-only trend grouped by agent/model/host/location
kql_runs_trend = r"""
let lookback = 6h;
AppDependencies
| where TimeGenerated > ago(lookback)
| where Name in ("agent-query", "responses") or Name startswith "chat "
| extend
    agent_name = coalesce(
        tostring(Properties["agent.name"]),
        tostring(split(tostring(Properties["gen_ai.agent.id"]), ":")[0]),
        ""
    ),
    model_used = coalesce(
        tostring(Properties["gen_ai.request.model"]),
        tostring(Properties["gen_ai.response.model"]),
        ""
    ),
    computer = tostring(AppRoleInstance),
    location = coalesce(
        tostring(Properties["_workspace_location"]),
        tostring(Properties["cloud.region"]),
        ""
    )
| summarize
    Calls = count(),
    Failures = countif(Success == false),
    AvgDurationMs = avg(DurationMs),
    P95DurationMs = percentile(DurationMs, 95)
    by bin(TimeGenerated, 15m), agent_name, model_used, computer, location
| order by TimeGenerated desc
""".strip()

def query_log_analytics(customer_id: str, query: str) -> dict:
    credential = DefaultAzureCredential()
    token = credential.get_token("https://api.loganalytics.io/.default").token

    body = json.dumps({"query": query}).encode("utf-8")
    request = urllib.request.Request(
        url=f"https://api.loganalytics.io/v1/workspaces/{customer_id}/query",
        data=body,
        method="POST",
        headers={
            "Content-Type": "application/json",
            "Authorization": f"Bearer {token}",
        },
    )

    try:
        with urllib.request.urlopen(request, timeout=60) as response:
            return json.loads(response.read().decode("utf-8"))
    except urllib.error.HTTPError as ex:
        error_body = ex.read().decode("utf-8", errors="replace")
        raise RuntimeError(f"Log Analytics query failed ({ex.code}): {error_body}") from ex

print("Running end-to-end validation query...")
end_to_end_result = query_log_analytics(workspace_customer_id, kql_end_to_end)
print(json.dumps(end_to_end_result, indent=2)[:4000])

print("\nRunning runs-only trend query...")
runs_trend_result = query_log_analytics(workspace_customer_id, kql_runs_trend)
print(json.dumps(runs_trend_result, indent=2)[:4000])